# 🔥 Firecrawl Demo - Web Scraping for AI/LLM

**Firecrawl** là một web scraping API được tối ưu hóa cho AI và LLM. Notebook này demo các feature chính của Firecrawl SDK.

| Feature | Mô tả |
|---------|-------|
| **Scrape** | Lấy nội dung từ 1 URL |
| **Crawl** | Crawl toàn bộ domain |
| **Map** | Khám phá URLs theo topic |
| **Extract** | Trích xuất dữ liệu có cấu trúc (AI-powered) |

## 1. Setup & Installation

In [ ]:
# Cài đặt Firecrawl SDK
% pip install -q firecrawl-py pydantic

## 2. API Key Configuration

### Lấy API Key
1. Truy cập [firecrawl.dev](https://firecrawl.dev)
2. Đăng ký tài khoản (có **Free Tier** với 500 credits/tháng)
3. Copy API key từ Dashboard

### Cấu hình trên Colab
Để bảo mật API key, sử dụng **Colab Secrets**:
1. Click icon 🔑 ở thanh bên trái
2. Thêm secret với tên `FIRECRAWL_API_KEY`
3. Paste API key của bạn

In [ ]:
import os

# Cách 1: Dùng Colab Secrets
try:
    from google.colab import userdata
    FIRECRAWL_API_KEY = userdata.get('FIRECRAWL_API_KEY')
    print("Loaded API key from Colab Secrets")
except:
    # Cách 2: Nhập trực tiếp (không khuyến khích)
    FIRECRAWL_API_KEY = input("Enter your Firecrawl API key: ")
    print("Consider using Colab Secrets for security")

assert FIRECRAWL_API_KEY, "API key is required!"


Loaded API key from Colab Secrets


In [ ]:
from firecrawl import FirecrawlApp

# Khởi tạo Firecrawl App
app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)
print("🔥 Firecrawl initialized!")


🔥 Firecrawl initialized!


---
## 3. Scrape - Lấy nội dung từ một URL

Đây là feature cơ bản nhất - lấy nội dung từ một trang web và chuyển đổi sang định dạng phù hợp cho AI/LLM.

In [ ]:
# Demo: Scrape một trang Wikipedia
url = "https://vi.wikipedia.org/wiki/Hà_Nội"

result = app.scrape(
    url=url,
    formats=["markdown", "summary"],  # Output formats
    only_main_content=True,  # Chỉ lấy nội dung chính, bỏ navbar, footer, ads...
)

if result is None:
    print("Scrape operation returned None. The URL might be unreachable or Firecrawl had an issue.")
else:
    for attr in dir(result):
        if not attr.startswith('_'):
            attr_value = getattr(result, attr)
            print(f"  • {attr}: {str(attr_value)[:500]}...")


  • actions: None...
  • branding: None...
  • change_tracking: None...
  • construct: <bound method BaseModel.construct of <class 'firecrawl.v2.types.Document'>>...
  • copy: <bound method BaseModel.copy of Document(markdown='[Bước tới nội dung](https://vi.wikipedia.org/wiki/H%C3%A0_N%E1%BB%99i#bodyContent)\n\n|     |     |\n| --- | --- |\n| **Biểu quyết nội dung chọn lọc** **[Bài viết chọn lọc](https://vi.wikipedia.org/wiki/Wikipedia:%E1%BB%A8ng_c%E1%BB%AD_vi%C3%AAn_b%C3%A0i_vi%E1%BA%BFt_ch%E1%BB%8Dn_l%E1%BB%8Dc "Wikipedia:Ứng cử viên bài viết chọn lọc")**: [Trận Prokhorovka](https://vi.wikipedia.org/wiki/Wikipedia:%E1%BB%A8ng_c%E1%BB%AD_vi%C3%AAn_b%C3%A0i_vi%E1%BA%...
  • dict: <bound method BaseModel.dict of Document(markdown='[Bước tới nội dung](https://vi.wikipedia.org/wiki/H%C3%A0_N%E1%BB%99i#bodyContent)\n\n|     |     |\n| --- | --- |\n| **Biểu quyết nội dung chọn lọc** **[Bài viết chọn lọc](https://vi.wikipedia.org/wiki/Wikipedia:%E1%BB%A8ng_c%E1%BB%AD_vi%C3%AAn_b%C3%A0i_vi%E

/tmp/ipython-input-1879219688.py:15: PydanticDeprecatedSince211: Accessing the 'model_computed_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  attr_value = getattr(result, attr)
/tmp/ipython-input-1879219688.py:15: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  attr_value = getattr(result, attr)


In [ ]:
import re

def clean_markdown(md):
    # 1. Xóa links[text](url)
    md = re.sub(r'\[([^\]]+)\]\([^)]*\)', r'\1', md)
    # 2. Xóa image links: ![alt](url)
    md = re.sub(r'!\[([^\]]*)\]\([^)]*\)', r'\1', md)
    # 3. Xóa HTML <a href=""> tags
    md = re.sub(r'<a\s+[^>]*>(.*?)</a>', r'\1', md, flags=re.DOTALL)
    # 4. Xóa URLs: http://..., https://...
    md = re.sub(r'https?://\S+', '', md)
    # 5. Xóa các tags: <br>
    md = re.sub(r'<br>', '', md)
    # 6. Xóa ký tự đặc biệt: * và ^ bằng regex
    md = re.sub(r'\*|\^', '', md)
    return md.strip()


In [ ]:
print(result.markdown[4115:10000])

Hà Nội |
| --- |
| [Thành phố trực thuộc trung ương](https://vi.wikipedia.org/wiki/Th%C3%A0nh_ph%E1%BB%91_tr%E1%BB%B1c_thu%E1%BB%99c_trung_%C6%B0%C6%A1ng_(Vi%E1%BB%87t_Nam) "Thành phố trực thuộc trung ương (Việt Nam)") |
| [![](https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Emblem_of_Hanoi.svg/120px-Emblem_of_Hanoi.svg.png)](https://vi.wikipedia.org/wiki/T%E1%BA%ADp_tin:Emblem_of_Hanoi.svg)<br>Biểu trưng |
| [![](https://upload.wikimedia.org/wikipedia/commons/thumb/1/10/Hanoi_Skyline_-_NKS.jpg/330px-Hanoi_Skyline_-_NKS.jpg)](https://vi.wikipedia.org/wiki/T%E1%BA%ADp_tin:Hanoi_Skyline_-_NKS.jpg)<br>Quang cảnh thành phố nhìn từ [cầu Nhật Tân](https://vi.wikipedia.org/wiki/C%E1%BA%A7u_Nh%E1%BA%ADt_T%C3%A2n "Cầu Nhật Tân")<br>[![](https://upload.wikimedia.org/wikipedia/commons/thumb/f/fd/L%C4%83ng_B%C3%A1c_-_NKS.jpg/250px-L%C4%83ng_B%C3%A1c_-_NKS.jpg)](https://vi.wikipedia.org/wiki/T%E1%BA%ADp_tin:L%C4%83ng_B%C3%A1c_-_NKS.jpg)<br>[Lăng Bác](https://vi.wikipedia.org/wiki/L%C4%83n

In [ ]:
print(f"Summary: {result.summary}")
cleaned_markdown = clean_markdown(result.markdown)
print(f"Content: {cleaned_markdown}")

Summary: Hà Nội, thủ đô của Việt Nam, là thành phố lớn thứ hai về dân số và được xếp loại đô thị đặc biệt. Nằm ở phía Tây Bắc của đồng bằng châu Thổ sông Hồng, Hà Nội có diện tích 3.359,82 km² với dân số gần 8,5 triệu người vào năm 2025. Thành phố này được coi là trung tâm chính trị, văn hóa và kinh tế của Việt Nam, với bề dày lịch sử và nhiều địa điểm văn hóa nổi tiếng. Hà Nội từng mang nhiều tên gọi và trải qua nhiều thời kỳ lịch sử quan trọng, từ Thăng Long, Đông Đô đến Đông Kinh. Trong suốt quá trình phát triển, Hà Nội bị ảnh hưởng bởi nhiều nền văn hóa, đặc biệt là khi trở thành thuộc địa của Pháp. Ngày nay, Hà Nội không chỉ lưu giữ nhiều di sản văn hóa mà còn là nền tảng cho sự phát triển kinh tế và đô thị hóa mạnh mẽ.
Content: Bước tới nội dung

|     |     |
| --- | --- |
| Biểu quyết nội dung chọn lọc Bài viết chọn lọc: Trận Prokhorovka • Báo động khẩn, tình yêu hạ cánh • Grand Theft Auto V (lần 2) "Wikipedia:Ứng cử viên bài viết chọn lọc/Grand Theft Auto V (lần 2)") • Hanoi's


| Option | Mô tả |
|--------|-------|
| `formats` | `["markdown", "html", "links", "screenshot"]` |
| `only_main_content` | Bỏ qua navigation, footer, sidebar |
| `include_tags` | Chỉ lấy content từ các tags cụ thể |
| `exclude_tags` | Loại bỏ content từ các tags cụ thể |

In [ ]:
# Demo: Scrape với nhiều options
url = "https://vnexpress.net/du-lich/am-thuc"

result = app.scrape(
    url=url,
    formats=["markdown", "links", "summary"],
    only_main_content=True,
    exclude_tags=["nav", "header", "footer", "aside"],
)

print(f"Scraped: {url}")
print(f"Content: {len(result.markdown)} chars")
print(f"Found {len(result.links) if hasattr(result, 'links') else 0} links")

if result is None:
    print("Scrape operation returned None. The URL might be unreachable or Firecrawl had an issue.")
else:
    for attr in dir(result):
        if not attr.startswith('_'):
            attr_value = getattr(result, attr)
            print(f"  • {attr}: {str(attr_value)[:500]}...")

Scraped: https://vnexpress.net/du-lich/am-thuc
Content: 46860 chars
Found 232 links
  • actions: None...
  • branding: None...
  • change_tracking: None...
  • construct: <bound method BaseModel.construct of <class 'firecrawl.v2.types.Document'>>...
  • copy: <bound method BaseModel.copy of Document(markdown='Tất cả chuyên mụcĐóng\n\n- [VnE-GO](https://vnexpress.net/vne-go "VnE-GO")\n- [Discover](https://vnexpress.net/vne-go/discover "Discover")\n- [Shorts](https://vnexpress.net/vne-go "Shorts")\n- [Podcasts](https://vnexpress.net/vne-go/podcast "Podcasts")\n\n- [Thời sự](https://vnexpress.net/thoi-su "Thời sự")\n- [Chính trị](https://vnexpress.net/thoi-su/chinh-tri "Chính trị")\n- [Nhân sự](https://vnexpress.net/thoi-su/chinh-tri/nhan-su "Nhân sự")\...
  • dict: <bound method BaseModel.dict of Document(markdown='Tất cả chuyên mụcĐóng\n\n- [VnE-GO](https://vnexpress.net/vne-go "VnE-GO")\n- [Discover](https://vnexpress.net/vne-go/discover "Discover")\n- [Shorts](https://vnexpress.net/vn

/tmp/ipython-input-595496237.py:20: PydanticDeprecatedSince211: Accessing the 'model_computed_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  attr_value = getattr(result, attr)
/tmp/ipython-input-595496237.py:20: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  attr_value = getattr(result, attr)


In [ ]:
print(f"Summary: {result.summary}")
print(f"Content: {result.markdown[:1000]}")
print(f"Link: {result.links[:10]} links")


Summary: The content provides a comprehensive overview of various articles featured on VnExpress related to food, culture, and lifestyle trends in Vietnam and beyond. It highlights unique dishes and culinary experiences, including the specialty 'nhút Thanh Chương' from Nghệ An and 'tiết canh vịt,' which has gained international recognition. The platform also discusses cultural insights, trends in coffee consumption, and notable Vietnamese eateries that have garnered attention worldwide. Moreover, it emphasizes the culinary diversity of Vietnam, encouraging tourists and locals to explore regional favorites and traditional recipes.
Content: Tất cả chuyên mụcĐóng

- [VnE-GO](https://vnexpress.net/vne-go "VnE-GO")
- [Discover](https://vnexpress.net/vne-go/discover "Discover")
- [Shorts](https://vnexpress.net/vne-go "Shorts")
- [Podcasts](https://vnexpress.net/vne-go/podcast "Podcasts")

- [Thời sự](https://vnexpress.net/thoi-su "Thời sự")
- [Chính trị](https://vnexpress.net/thoi-su/chinh-t

---
## 4. Crawl - Crawl toàn bộ Website

Crawl đệ quy từ URL gốc, thu thập tất cả các trang con.

In [ ]:
# Demo: Crawl một website với giới hạn số trang
url = "https://vnpt.vn/goi-home"

result = app.crawl(
    url=url,
    limit=10,  # Giới hạn trang
    scrape_options={
        "formats": ["markdown", "summary"],
        "only_main_content": True,
    }
)

if result is None:
    print("Crawled operation returned None. The URL might be unreachable or Firecrawl had an issue.")
else:
    for attr in dir(result):
        if not attr.startswith('_'):
            attr_value = getattr(result, attr)
            print(f"  • {attr}: {str(attr_value)[:500]}...")


  • completed: 10...
  • construct: <bound method BaseModel.construct of <class 'firecrawl.v2.types.CrawlJob'>>...
  • copy: <bound method BaseModel.copy of CrawlJob(status='completed', total=10, completed=10, credits_used=10, expires_at=datetime.datetime(2025, 12, 6, 19, 23, 10, tzinfo=TzInfo(0)), next=None, data=[Document(markdown='[![Tập đoàn Bưu chính Viễn thông Việt Nam](https://vnpt.vn/Design/images/logo-vnpt-app.jpg)](https://vnpt.vn/)\n\nMyVNPT: Nạp thẻ qua app, nhận quà khuyến mại\n\nỨng dụng tiện ích, tra cứu tích điểm\n\n[Tải ngay](http://onelink.to/qdgev3)\n\n[![Tập đoàn Bưu chính Viễn thông Việt Nam](http...
  • credits_used: 10...
  • data: [Document(markdown='[![Tập đoàn Bưu chính Viễn thông Việt Nam](https://vnpt.vn/Design/images/logo-vnpt-app.jpg)](https://vnpt.vn/)\n\nMyVNPT: Nạp thẻ qua app, nhận quà khuyến mại\n\nỨng dụng tiện ích, tra cứu tích điểm\n\n[Tải ngay](http://onelink.to/qdgev3)\n\n[![Tập đoàn Bưu chính Viễn thông Việt Nam](https://vnpt.vn/design/images/lo

/tmp/ipython-input-1434365533.py:18: PydanticDeprecatedSince211: Accessing the 'model_computed_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  attr_value = getattr(result, attr)
/tmp/ipython-input-1434365533.py:18: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  attr_value = getattr(result, attr)


In [ ]:
for idx, page in enumerate(result.data):
    print(idx, page.markdown[5000:5500])


0 không chỉ nhận được ưu đãi lên tới 50% chi phí so với lắp đặt các dịch vụ riêng lẻ mà còn được sử dụng các dịch vụ nội dung chất lượng quốc
tế như gói truyền hình MyTV gần 200 kênh (bao gồm HD và SD) như HBO, Cinemax, Fox Sport…, đường truyền internet lên tới 300Mbps, miễn phí thoại và chia sẻ data nội nhóm….
Đăng ký ngay Home Combo để nhận nhiều ưu đãi hấp dẫn


![Home Đỉnh](https://media-vnpt.vnptvas.vn/Media/Images/upload_images/images/202503/img_vm_2503311603432481.jpg?w=490&mode=crop)

### 
1 g)[Thông tin liên hệ](https://vnpt.vn/lien-he "KHS")

![Điểm giao dịch](https://vnpt.vn/design/images/support3.jpg)[Điểm giao dịch](https://vnpt.vn/ho-tro/diem-giao-dich "ĐGD")

![ Thời gian giữ số](https://vnpt.vn/design/images/support4.jpg)[Thời gian giữ số](https://vnpt.vn/di-dong/chinh-sach "ĐGD")

![Ứng dụng](https://vnpt.vn/design/images/icon-phone.png)Các ứng dụng

![Vinaphone Plus](https://vnpt.vn/design/images/logo-vinphone-plus.jpg)

VinaPhone Plus

[Download](https://vnpt.vn/goi-

### Crawl Options

| Option | Mô tả |
|--------|-------|
| `limit` | Số trang tối đa |
| `include_paths` | Chỉ crawl các path khớp pattern (vd: `["/blog/*"]`) |
| `exclude_paths` | Bỏ qua các path (vd: `["/admin/*"]`) |
| `allowed_domains` | Danh sách domain được phép |

In [ ]:
# Demo: Crawl với path filter
url = "https://vi.wikipedia.org/wiki/Việt_Nam"

result = app.crawl(
    url=url,
    limit=3,
    include_paths=["/wiki/*"],  # Chỉ crawl các trang wiki
    exclude_paths=["/wiki/Special:*", "/wiki/Wikipedia:*"],  # Bỏ qua trang đặc biệt
    scrape_options={
        "formats": ["markdown", "summary"],
        "only_main_content": True,
    }
)

pages = result.data if hasattr(result, 'data') else []
print(f"Crawled {len(pages)} Wikipedia pages")
for page in pages:
    if hasattr(page, 'metadata') and hasattr(page.metadata, 'sourceURL'):
        print(f"  • {page.metadata.sourceURL}")


Crawled 2 Wikipedia pages


In [ ]:
for idx, page in enumerate(pages):
    print(idx, page.markdown[4000:6000], end="\n\n")

0 ung chú thích tới [các nguồn đáng tin cậy](https://vi.wikipedia.org/wiki/Wikipedia:Ngu%E1%BB%93n_%C4%91%C3%A1ng_tin_c%E1%BA%ADy "Wikipedia:Nguồn đáng tin cậy"). Các nội dung không có nguồn có thể bị nghi ngờ và xóa bỏ._([Tìm hiểu cách thức và thời điểm xóa thông báo này](https://en.wikipedia.org/wiki/Help:Maintenance_template_removal "en:Help:Maintenance template removal"))_ |

| Việt Nam tại<br>Đại hội Thể thao Đông Nam Á 2005 |
| --- |
| [![](https://upload.wikimedia.org/wikipedia/commons/thumb/2/21/Flag_of_Vietnam.svg/250px-Flag_of_Vietnam.svg.png)](https://vi.wikipedia.org/wiki/T%E1%BA%ADp_tin:Flag_of_Vietnam.svg) |
| [Mã IOC](https://vi.wikipedia.org/wiki/B%E1%BA%A3ng_m%C3%A3_IOC "Bảng mã IOC") | VIE |
| [NOC](https://vi.wikipedia.org/wiki/%E1%BB%A6y_ban_Olympic_qu%E1%BB%91c_gia "Ủy ban Olympic quốc gia") | [Ủy ban Olympic Việt Nam](https://vi.wikipedia.org/wiki/%E1%BB%A6y_ban_Olympic_Vi%E1%BB%87t_Nam "Ủy ban Olympic Việt Nam") |
| Website | [www.voc.org.vn](http://www.voc.org.v

---
## 5. Map - Khám phá URLs theo Topic

Tìm kiếm và trả về danh sách URLs liên quan đến một chủ đề cụ thể.

In [ ]:
# Demo: Map URLs theo topic
url = "https://vi.wikipedia.org"
topic = "lịch sử Việt Nam"

result = app.map(
    url=url,
    search=topic,
    limit=100,
)

# Lấy danh sách URLs
urls = [link.url if hasattr(link, 'url') else link for link in result.links]

print(f"Found {len(urls)} URLs related to '{topic}'")
for i, url in enumerate(urls, 1):
    print(f"  {i}. {url}")


Found 82 URLs related to 'lịch sử Việt Nam'
  1. https://vi.wikipedia.org/wiki/B%E1%BA%A3n_m%E1%BA%ABu:Ch%E1%BB%A7_%C4%91%E1%BB%81_L%E1%BB%8Bch_s%E1%BB%AD_Vi%E1%BB%87t_Nam
  2. https://vi.wikipedia.org/wiki/Th%E1%BB%83_lo%E1%BA%A1i:S%C3%A1ch_l%E1%BB%8Bch_s%E1%BB%AD_Vi%E1%BB%87t_Nam
  3. https://vi.wikipedia.org/wiki/Ni%C3%AAn_bi%E1%BB%83u_l%E1%BB%8Bch_s%E1%BB%AD_Vi%E1%BB%87t_Nam
  4. https://vi.wikipedia.org/wiki/B%E1%BA%A3n_m%E1%BA%ABu:L%E1%BB%8Bch_s%E1%BB%AD_Vi%E1%BB%87t_Nam
  5. https://vi.wikipedia.org/wiki/L%E1%BB%8Bch_s%E1%BB%AD_qu%E1%BB%91c_k%E1%BB%B3_Vi%E1%BB%87t_Nam
  6. https://vi.wikipedia.org/wiki/L%E1%BB%8Bch_s%E1%BB%AD_Vi%E1%BB%87t_Nam
  7. https://vi.wikipedia.org/wiki/H%E1%BB%99i_Khoa_h%E1%BB%8Dc_L%E1%BB%8Bch_s%E1%BB%AD_Vi%E1%BB%87t_Nam
  8. https://vi.wikipedia.org/wiki/Th%E1%BB%83_lo%E1%BA%A1i:B%C3%A0i_ch%C6%B0a_x%E1%BA%BFp_lo%E1%BA%A1i_ch%E1%BA%A5t_l%C6%B0%E1%BB%A3ng_v%E1%BB%81_L%E1%BB%8Bch_s%E1%BB%AD_Vi%E1%BB%87t_Nam
  9. https://vi.wikipedia.org/wiki/B%E1%BA%A3n_m%

### Map + Scrape Combo

Kết hợp Map để tìm URLs, sau đó Scrape từng trang.

In [ ]:
import time

# Demo: Map + Scrape workflow
base_url = "https://vi.wikipedia.org"
topic = "văn hóa Việt Nam"

# Step 1: Map để tìm URLs
print(f" Mapping URLs for topic: '{topic}'...")
map_result = app.map(url=base_url, search=topic, limit=3)

urls_to_scrape = []
if hasattr(map_result, 'links'):
    urls_to_scrape = [link.url if hasattr(link, 'url') else link for link in map_result.links[:3]]
elif isinstance(map_result, list):
    urls_to_scrape = map_result[:3]

print(f"Found {len(urls_to_scrape)} URLs to scrape")

# Step 2: Scrape từng URL
documents = []
for url in urls_to_scrape:
    print(f"Scraping: {url[:60]}...")
    try:
        result = app.scrape(url, formats=["markdown"], only_main_content=True)
        if hasattr(result, 'markdown') and result.markdown:
            documents.append({
                "url": url,
                "title": result.metadata.title if hasattr(result, 'metadata') else "",
                "content": result.markdown[5000:6000],  # Preview only
            })
        time.sleep(2)  # Rate limiting
    except Exception as e:
        print(f"     Error: {e}")

print(f"\n Scraped {len(documents)} documents")
for doc in documents:
    print(f"\n {doc['title']}")
    print(f"   {doc['content'][:200]}...")


 Mapping URLs for topic: 'văn hóa Việt Nam'...
Found 3 URLs to scrape
Scraping: https://vi.wikipedia.org/wiki/V%C4%83n_h%C3%B3a_Vi%E1%BB%87t...
Scraping: https://vi.wikipedia.org/wiki/Th%E1%BB%83_lo%E1%BA%A1i:V%C4%...
Scraping: https://vi.wikipedia.org/wiki/Vi%E1%BB%87t_Nam...

 Scraped 3 documents

 Văn hóa Việt Nam – Wikipedia tiếng Việt
    Việt.
- Quan niệm thứ hai: Văn hóa Việt Nam là toàn bộ văn hóa [các dân tộc Việt Nam](https://vi.wikipedia.org/wiki/D%C3%A2n_t%E1%BB%99c_Vi%E1%BB%87t_Nam "Dân tộc Việt Nam") cư trú trên mảnh đất Việt...

 Thể loại:Văn hóa Việt Nam – Wikipedia tiếng Việt
   óa, Thể thao và Du lịch Việt Nam")(3 t.l., 2 tr.)


### C

- [Các dân tộc Việt Nam](https://vi.wikipedia.org/wiki/Th%E1%BB%83_lo%E1%BA%A1i:C%C3%A1c_d%C3%A2n_t%E1%BB%99c_Vi%E1%BB%87t_Nam "Thể loại:Các ...

 Việt Nam – Wikipedia tiếng Việt
   rojection).svg)<br>[![](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d6/Location_Vietnam_ASEAN.svg/250px-Location_Vietnam_ASEAN.svg.png)](https://

---
## 6. Extract - Trích xuất dữ liệu có cấu trúc (AI-powered)

Feature mạnh mẽ nhất của Firecrawl - sử dụng LLM để tự động trích xuất dữ liệu theo schema định trước.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Định nghĩa schema cho dữ liệu cần extract
class ArticleInfo(BaseModel):
    """Schema cho thông tin bài viết"""
    title: str = Field(description="Tiêu đề bài viết")
    summary: str = Field(description="Tóm tắt nội dung chính trong 2-3 câu")
    key_points: List[str] = Field(description="Các điểm chính của bài viết")
    topics: List[str] = Field(description="Các chủ đề liên quan")

print("Schema defined!")
print(ArticleInfo.model_json_schema())


Schema defined!
{'description': 'Schema cho thông tin bài viết', 'properties': {'title': {'description': 'Tiêu đề bài viết', 'title': 'Title', 'type': 'string'}, 'summary': {'description': 'Tóm tắt nội dung chính trong 2-3 câu', 'title': 'Summary', 'type': 'string'}, 'key_points': {'description': 'Các điểm chính của bài viết', 'items': {'type': 'string'}, 'title': 'Key Points', 'type': 'array'}, 'topics': {'description': 'Các chủ đề liên quan', 'items': {'type': 'string'}, 'title': 'Topics', 'type': 'array'}}, 'required': ['title', 'summary', 'key_points', 'topics'], 'title': 'ArticleInfo', 'type': 'object'}


In [ ]:
# Demo: Extract thông tin từ trang Wikipedia
url = "https://vi.wikipedia.org/wiki/Hà_Nội"

result = app.extract(
    urls=[url],
    schema=ArticleInfo.model_json_schema(), # Pass the JSON schema
)


In [ ]:
print("AI Extracted Data:")
print(f"\nTitle: {result.data['title']}")
print(f"\nSummary: {result.data['summary']}")
print(f"\nKey Points: {result.data['key_points']}")
print(f"\nTopics: {result.data['topics']}")


AI Extracted Data:

Title: Hà Nội

Summary: Hà Nội là thủ đô của Việt Nam, nổi tiếng với lịch sử lâu dài và văn hóa phong phú. Thành phố có diện tích 3.359,82 km² và dân số khoảng 8,8 triệu người vào năm 2025.

Key Points: ['Thành phố trực thuộc trung ương', 'Diện tích: 3.359,82 km²', 'Dân số: 8.807.523 người (2025)', 'Mật độ dân số: 2.621 người/km²', 'Kinh tế: GRDP 1.195.989 tỉ đồng (2022)']

Topics: ['Địa lý', 'Lịch sử', 'Văn hóa', 'Kinh tế', 'Xã hội', 'Giao thông']


### Extract với Prompt (không schema)

In [ ]:
# Demo: Extract với prompt
url = "https://vnexpress.net"

result = app.extract(
    urls=[url],
    prompt="Liệt kê 10 tin tức về du lịch nổi bật nhất trên trang, kèm theo category của từng tin."
)

print("AI Extracted Headlines:")
print(result.data)


AI Extracted Headlines:
{'news': [{'title': 'Cơ hội hút khách cao cấp khi Việt Nam thêm 41 cửa khẩu e-visa', 'category': 'Du lịch'}, {'title': "Chuyển công năng bến du thuyền Vũ 'Nhôm' sang mục đích công cộng", 'category': 'Du lịch'}, {'title': 'Tiết canh vịt vào danh sách món ăn từ vịt ngon nhất thế giới', 'category': 'Du lịch'}, {'title': 'Nhút Thanh Chương là món gì?', 'category': 'Du lịch'}, {'title': 'Hà Nội vận hành hơn 1.800 camera AI từ 10/12', 'category': ''}, {'title': 'Xuất hiện ảnh tàu ba thân bí ẩn của Trung Quốc', 'category': ''}, {'title': 'Lập quy hoạch bảo tồn đền thờ danh tướng Trần Quốc Tảng', 'category': ''}, {'title': 'Trục vớt hàng chục tàu bị lũ nhấn chìm ở Lâm Đồng', 'category': ''}, {'title': '9 người Campuchia thiệt mạng trong tai nạn ở Thái Lan', 'category': ''}, {'title': 'Mâu thuẫn nội bộ khiến đơn vị Ukraine ở Kiev bị Nga tập kích', 'category': ''}]}


### Ví dụ: Extract thông tin về tài liệu luật

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field

# Schema cho văn bản pháp luật / nội dung pháp lý Việt Nam
class VietnamLawInfo(BaseModel):
    title: str = Field(description="Tên văn bản / tiêu đề nội dung pháp luật")
    doc_type: Optional[str] = Field(description="Loại văn bản (Luật, Nghị định, Thông tư, Nghị quyết, Quyết định...)")
    number: Optional[str] = Field(description="Số, ký hiệu văn bản (ví dụ: 15/2023/QH15)")
    issuing_authority: Optional[str] = Field(description="Cơ quan ban hành (Quốc hội, Chính phủ, Bộ Tài chính...)")
    issue_date: Optional[str] = Field(description="Ngày ban hành")
    effective_date: Optional[str] = Field(description="Ngày có hiệu lực")
    status: Optional[str] = Field(description="Tình trạng hiệu lực (còn hiệu lực/hết hiệu lực/bị thay thế/sửa đổi...)")

    scope: Optional[str] = Field(description="Phạm vi điều chỉnh / đối tượng áp dụng (nếu có)")
    key_points: List[str] = Field(default_factory=list, description="Các điểm chính / nội dung nổi bật")
    prohibited_acts: List[str] = Field(default_factory=list, description="Các hành vi bị cấm (nếu có)")
    obligations: List[str] = Field(default_factory=list, description="Nghĩa vụ/quy định cần tuân thủ")
    penalties: List[str] = Field(default_factory=list, description="Chế tài/xử phạt/mức phạt (nếu có)")

    referenced_documents: List[str] = Field(default_factory=list, description="Văn bản liên quan/được viện dẫn")
    source_url: Optional[str] = Field(description="URL nguồn của trang trích xuất")


law_url = "https://vbpl.vn/botuphap/Pages/vbpq-toanvan.aspx?ItemID=95942&dvid=41"

result = app.extract(urls=[law_url], schema=VietnamLawInfo.model_json_schema())
print("AI Extracted Vietnam Law Info:")
print(result.data)


AI Extracted Vietnam Law Info:
{'title': 'Bộ luật Dân sự 91/2015/QH13', 'doc_type': 'Luật', 'number': '91/2015/QH13', 'issuing_authority': 'Quốc hội', 'issue_date': '2015-11-24', 'effective_date': '2017-01-01', 'status': 'Còn hiệu lực', 'scope': '', 'key_points': ['Quy định địa vị pháp lý, chuẩn mực pháp lý về cách ứng xử của cá nhân, pháp nhân.', 'Công nhận, tôn trọng, bảo vệ và bảo đảm quyền dân sự.', 'Các nguyên tắc cơ bản của pháp luật dân sự.'], 'prohibited_acts': [], 'obligations': [], 'penalties': [], 'referenced_documents': [], 'source_url': 'https://vbpl.vn/botuphap/Pages/vbpq-toanvan.aspx?ItemID=95942&dvid=41'}
